# 01 · CEFR-SP — build the pool

*The on-ramp track: sentence proficiency level (A1–C2)*

### Where this sits

```
▶ 01 build the pool  →  02 sample  →  03 annotate  →  04 prompt  →  05 report
```

You run **01 once per group**, for your own track only. It ends by writing `data/pools/<track>_pool.json` — the file notebook 02 opens.

---

**What it is.** Sentences annotated with a CEFR level by two trained annotators. We use the openly-shipped **Wiki-Auto** portion.

**Difficulty of the labeling judgment:** ★☆☆ — easy. Levels are concrete and the annotators usually agree.

**Licence:** CC BY-SA 3.0 (Wiki-Auto portion) — **share-alike**, so anything you redistribute from it inherits the same licence.  
**Cite:** Arase, Uchida & Kajiwara (2022), *EMNLP*. github.com/yukiar/CEFR-SP

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

> The reshaping code below is read straight out of `scripts/reshape.py` — it is the same code `scripts/prep_datasets.py` runs, not a copy of it. What is *missing* from it is missing on purpose: the ✏️ cells are the decisions, and they are yours. (Generated by `scripts/_generate_pool_notebooks.py`; edit that or `reshape.py`, never the `.ipynb`.)

## Step 1 — Download the raw data

The corpus lives in a GitHub repository, so we clone it. (`!` runs a shell command from inside the notebook.)

In [ ]:
!git clone --depth 1 https://github.com/yukiar/CEFR-SP

## Step 2 — Look at the raw format

Note the folder path: cloning `CEFR-SP` gives you a `CEFR-SP` folder *inside* `CEFR-SP`. Easy to trip over.

The Wiki-Auto files are **tab-separated text**, one sentence per line:

```
sentence <TAB> label_by_annotator_A <TAB> label_by_annotator_B
```

Labels are digits: `1`=A1, `2`=A2, … `6`=C2. **Look at the output before you write anything below** — the mapping you are about to type has to match what is actually in the file.

In [ ]:
RAW_DIR = "CEFR-SP/CEFR-SP/Wiki-Auto"

with open(RAW_DIR + "/CEFR-SP_Wikiauto_dev.txt", encoding="utf-8") as f:
    for _ in range(5):
        print(repr(next(f)))

## Step 3 — Reshape into the canonical schema

Three decisions are baked into this track, and the first is yours to write:

1. ✏️ **What the levels are called.** The file says `1`; a prompt that says `A1` needs far less explaining than one that says `1`. You supply that mapping — and its keys are also a filter: a row whose digit is not in your mapping gets dropped, so leaving a level out silently removes it from your study.
2. **Trust only agreement.** Each sentence has *two* annotators, and the code below keeps a row only when both chose the same level (`label_a == label_b`). Every remaining label is then unambiguous — which is what makes this the gentle track, and also makes it easier than the data really is. Read that line and make sure you can say why it is there; keeping the disagreements would have meant deciding whose label wins.
3. **Wiki-Auto only** — the repo also ships a `SCoRE/` folder under a *non-commercial* licence, and we deliberately never read it. Notice `RAW_DIR` points at `Wiki-Auto` specifically rather than at the repo root: that is what keeps the two apart.

In [ ]:
# ✏️ Step 3a · Name the levels ───────────────────────────────────
# Goal      : map the digit in the file to the label a prompt can actually use.
# Shape     : CEFR_NUM = {"1": "A1", ...}   # keys are STRINGS — the file is text
#             six entries, one per level
# Produce   : CEFR_NUM (a dict)      ← later cells use this name
# Careful   : a digit you leave out is a level you silently DROP.
#             Check the counts in step 4 against what you expected.
# Why blank : this is the label set your prompt, your annotation sheet and
#             your confusion matrix all inherit. Put it in PLAN.md.

# ✏️ your code here


Now the reshaping function itself. It reads the `CEFR_NUM` you just defined — if the cell below raises `NameError: CEFR_NUM`, go back and run the one above.

In [ ]:
from pathlib import Path

def reid(items):
    """Renumber ids sequentially from 1, keeping the current order."""
    renumbered = []
    next_id = 1
    for item in items:
        new_item = dict(item)
        new_item["id"] = next_id
        renumbered.append(new_item)
        next_id = next_id + 1
    return renumbered

def reshape_cefr(wiki_auto_dir):
    """Read the Wiki-Auto TSV files and keep only the sentences both annotators agreed on.

    Each line is:  sentence <TAB> label_by_annotator_A <TAB> label_by_annotator_B
    with labels as digits, 1=A1 ... 6=C2.

    Three real decisions here:
      1. TRUST ONLY AGREEMENT. A sentence is kept only when both annotators chose the
         same level, so every label is unambiguous. That is what makes this the gentle
         on-ramp track - and it also means the track is easier than the data really is.
      2. HUMAN-READABLE LABELS. 1 -> A1, so a prompt can name the levels the way a
         person would.
      3. WIKI-AUTO ONLY. CEFR-SP also ships a SCoRE portion, but it is CC BY-NC-SA
         (non-commercial), so we deliberately do not touch it - see data/SOURCES.md.
    """
    source_dir = Path(wiki_auto_dir)
    rows = []
    # Sorted, so a rebuild reads the files in the same order and ids stay stable.
    for path in sorted(source_dir.glob("*.txt")):
        for line in path.read_text(encoding="utf-8").splitlines():
            parts = line.split("\t")
            if len(parts) < 3:
                continue
            text = parts[0].strip()
            label_a = parts[1].strip()
            label_b = parts[2].strip()
            if text and label_a == label_b and label_a in CEFR_NUM:
                rows.append({"id": 0, "text": text, "label": CEFR_NUM[label_a]})
    return reid(rows)

In [ ]:
rows = reshape_cefr(RAW_DIR)
print("kept", len(rows), "sentences where both annotators agreed")

## Step 4 — Check the label balance

Look at the counts before you trust anything downstream. This corpus is **heavily imbalanced** — B1 and B2 dominate, and A1/C2 are scarce.

In [ ]:
from collections import Counter

print("total items:", len(rows))
print("label counts:", dict(Counter(item["label"] for item in rows)))
rows[:3]        # peek at the first three reshaped items

In [ ]:
# ✏️ Step 4b · React to the balance ──────────────────────────────
# Goal      : decide what the counts you just printed mean for your study.
# Shape     : MIN_PER_CLASS = <the size of your SMALLEST class>
#             that is the ceiling on N_PER_CLASS in config.py — a balanced
#             sample cannot draw more from a class than the class has
# Produce   : MIN_PER_CLASS (an int)      ← later cells use this name
# Note      : the agreement filter costs you rows unevenly — the levels annotators argue about lose the most.
# Note      : if the rarest class is tiny, say so in PLAN.md. Merging it
#             away or living with fewer items are both defensible;
#             not noticing is not.

# ✏️ your code here


## Step 5 — Save it

In [ ]:
# Save the pool. Two places you might want it:
#   * this repo, if you cloned it:  "../data/pools/cefr_pool.json"
#   * your Google Drive, so it survives the Colab runtime resetting
import json, pathlib

OUT_FILE = "../data/pools/cefr_pool.json"

# In Colab WITHOUT the repo, uncomment these two to write straight to Drive:
# from google.colab import drive; drive.mount("/content/drive")
# OUT_FILE = "/content/drive/MyDrive/cefr_pool.json"

pathlib.Path(OUT_FILE).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", OUT_FILE)

## What you just built, and what happens to it

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is **not** your gold set, and its labels are **not** your labels: they are the original corpus authors' judgment, and you have not yet agreed with them about anything.

What those labels are for is narrow, and worth being precise about:

1. **Stratifying the draw** in notebook 02 — you cannot sample evenly across classes without knowing what the classes are.
2. **A comparison** in notebook 03 — once you have annotated blind and adjudicated, `compare_to_published` shows you every item where your group landed somewhere different. That gap is evidence, and one of the more interesting things you can put in a report.

They are never the answer key you score the model against. That file does not exist yet — you make it in notebook 03.

---

**Next:** set `TRACK = "cefr"` in `config.py`, then open `02_sample.ipynb`.